# How to classify a RGB drone dataset into vegetation types

## Introduction

## Set-up

### Load packages

In [ ]:
import pickle as pkl
import os
import rasterio as rio
import numpy as np
from classification_aux import classify_mission, load_site_data, feature_mission

### Load data

In [ ]:
site = 'F3-8A'
run_name = f'{site}_01'

# location of raw lichen data
data_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/2_labelling/lp'

# place where we stored chunked feature data
run_dir = f'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/runs/{run_name}'

if not os.path.exists(run_dir):
    os.makedirs(os.path.join(run_dir, 'composite'))

## Calculate features

In [ ]:
rasters = load_site_data(data_dir, site)
chunks = 8

feature_mission(rasters, chunks, run_dir)

## Choose classifier

In [ ]:
model_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/results/2a_classifier/RF_is/'
with open(model_dir+f'RF_{site}.pkl', 'rb') as file:      
    model = pkl.load(file) 

## Classify drone data

In [ ]:
blocks = 8
rgb_file = os.path.join(data_dir, site, f'{site}_transparent_mosaic_group1.tif')
run_dir = f'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/runs/{run_name}'

output = classify_mission(blocks=blocks, rgb_file=rgb_file, run_dir=run_dir, model=model)

## Save result as .tiff

In [ ]:
out_file = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/results/classified/'

with rio.open(rgb_file) as f:
    meta = f.meta
meta.update(count=1)

# Write a new .tif file using the metadata from the original file
with rio.open(out_file, 'w', **meta) as dst:
    dst.write(output, 1)